# Notebook 5 - Build City Master Datasets

## Airbnb Capstone | Madrid and Tokyo master data

**Inputs:** raw `listings`, `calendar`, `reviews`, and geodata files

This notebook rebuilds one master modelling dataset per city from the raw source files. The goal is to create a reliable source-of-truth table for each city before modelling and before the later Streamlit chatbot dataset.

### Structure
1. Setup and helper functions
2. Audit raw review compatibility
3. Build Madrid and Tokyo master datasets
4. Review audit output
5. Inspect attributes and files


---
## 1. Setup, Imports, and Configuration

In [1]:
# Optional Colab setup
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/Shareddrives/MBD_Capstone Project_KPMG/2. Code

In [2]:
import ast
import gzip
import math
import shutil
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd


def get_project_root() -> Path:
    """Find the CAPSTONE folder from any notebook subfolder."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        data_dir = candidate / "1. Data"
        if (data_dir / "master" / "airbnb_clean_final.csv").exists() or (data_dir / "raw" / "madrid_listings.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing the 1. Data folder. Check the notebook working directory.")


PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "1. Data"
RAW_DIR = DATA_DIR / "raw"
GEODATA_DIR = RAW_DIR / "geodata"
OUTPUT_DIR = DATA_DIR / "master"

JPY_TO_EUR = 0.0062
CHUNK_SIZE = 1_000_000

This cell imports the required libraries, finds the CAPSTONE project folder, and defines the shared data paths.

In [3]:
CITY_CONFIG = {
    "madrid": {
        "city_label": "Madrid",
        "currency": "EUR",
        "center_lat": 40.4168,
        "center_lon": -3.7038,
        "reviews_file": "madrid_reviews_2025-09-14.csv",
        "reviews_url": "https://data.insideairbnb.com/spain/comunidad-de-madrid/madrid/2025-09-14/data/reviews.csv.gz",
    },
    "tokyo": {
        "city_label": "Tokyo",
        "currency": "JPY",
        "center_lat": 35.6812,
        "center_lon": 139.7671,
        "reviews_file": "tokyo_reviews.csv",
    },
}

This city configuration tells the workflow which raw files belong to Madrid and Tokyo, including the corrected Madrid reviews file.

In [4]:
LISTING_COLUMNS = [
    "id",
    "last_scraped",
    "host_since",
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate",
    "host_is_superhost",
    "host_listings_count",
    "host_total_listings_count",
    "host_has_profile_pic",
    "host_identity_verified",
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms_text",
    "bedrooms",
    "beds",
    "amenities",
    "price",
    "minimum_nights",
    "maximum_nights",
    "has_availability",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "first_review",
    "last_review",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month",
    "instant_bookable",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",
]

BOOLEAN_COLUMNS = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "has_availability",
    "instant_bookable",
]

KEY_AMENITIES = {
    "has_wifi": ["wifi"],
    "has_kitchen": ["kitchen"],
    "has_air_conditioning": ["air conditioning", "ac - split type ductless system"],
    "has_washer": ["washer"],
    "has_dedicated_workspace": ["dedicated workspace"],
    "has_tv": ["tv", "hdtv"],
    "has_parking": ["free parking", "paid parking", "parking"],
    "has_elevator": ["elevator"],
    "has_heating": ["heating"],
    "has_self_checkin": ["self check-in", "lockbox", "keypad", "smart lock"],
}

These column lists define which listing fields are loaded, which boolean fields are parsed, and which amenities are converted into model-ready flags.

In [5]:
MODEL_ATTRIBUTE_META = [
    ("listing_id", "identifier", "Listings", "Audit/join key only; do not use as a model predictor."),
    ("city", "context", "Listings", "Identifies the city; constant within each city-specific model but useful for audit."),
    ("price_eur", "target", "Listings", "Nightly listing price converted to EUR; target variable."),
    ("log_price_eur", "target_transform", "Engineered", "Log target option for skewed price distributions."),
    ("neighbourhood_cleansed", "categorical_feature", "Listings/geodata", "Captures local market/location effects."),
    ("neighbourhood_group", "categorical_feature", "Listings/geodata", "Broader location grouping where available."),
    ("latitude", "numeric_feature", "Listings", "Fine-grained location signal."),
    ("longitude", "numeric_feature", "Listings", "Fine-grained location signal."),
    ("distance_to_center_km", "numeric_feature", "Engineered", "Proxy for centrality and tourist/business access."),
    ("is_central_5km", "binary_feature", "Engineered", "Simple central-vs-noncentral location flag."),
    ("neighbourhood_listing_count", "numeric_feature", "Engineered", "Proxy for local Airbnb market density."),
    ("property_type", "categorical_feature", "Listings", "Captures property format and quality differences."),
    ("property_group", "categorical_feature", "Engineered", "Simplifies sparse property-type categories."),
    ("room_type", "categorical_feature", "Listings", "Major pricing driver: entire home/private/shared/hotel room."),
    ("accommodates", "numeric_feature", "Listings", "Strong capacity/size driver of price."),
    ("bedrooms", "numeric_feature", "Listings", "Property size signal."),
    ("beds", "numeric_feature", "Listings", "Property size signal."),
    ("bathrooms", "numeric_feature", "Engineered from listings", "Property size/comfort signal."),
    ("bathroom_type", "categorical_feature", "Engineered from listings", "Distinguishes private/shared bathroom setups."),
    ("amenities_count", "numeric_feature", "Engineered from listings", "Broad property-quality/value signal."),
    ("has_wifi", "binary_feature", "Engineered from amenities", "Amenity availability can affect guest willingness to pay."),
    ("has_kitchen", "binary_feature", "Engineered from amenities", "Important for longer stays and family/group bookings."),
    ("has_air_conditioning", "binary_feature", "Engineered from amenities", "Comfort feature, especially relevant seasonally."),
    ("has_washer", "binary_feature", "Engineered from amenities", "Useful for longer stays."),
    ("has_dedicated_workspace", "binary_feature", "Engineered from amenities", "Business/remote-work value signal."),
    ("has_tv", "binary_feature", "Engineered from amenities", "Basic comfort amenity."),
    ("has_parking", "binary_feature", "Engineered from amenities", "Can affect value in car-dependent areas."),
    ("has_elevator", "binary_feature", "Engineered from amenities", "Accessibility/convenience signal."),
    ("has_heating", "binary_feature", "Engineered from amenities", "Basic comfort amenity."),
    ("has_self_checkin", "binary_feature", "Engineered from amenities", "Convenience/operations signal."),
    ("host_response_time", "categorical_feature", "Listings", "Host service responsiveness signal."),
    ("host_response_rate", "numeric_feature", "Listings", "Host service responsiveness signal."),
    ("host_acceptance_rate", "numeric_feature", "Listings", "Host acceptance behaviour signal."),
    ("host_is_superhost", "binary_feature", "Listings", "Trust/quality signal."),
    ("host_identity_verified", "binary_feature", "Listings", "Trust signal."),
    ("host_tenure_days", "numeric_feature", "Engineered from listings", "Host experience signal."),
    ("host_listings_count", "numeric_feature", "Listings", "Professionalization/host scale signal."),
    ("is_professional_host", "binary_feature", "Engineered from listings", "Flags hosts with multiple listings."),
    ("calculated_host_listings_count", "numeric_feature", "Listings", "Host local portfolio size."),
    ("calculated_host_listings_count_entire_homes", "numeric_feature", "Listings", "Host portfolio composition."),
    ("instant_bookable", "binary_feature", "Listings", "Booking friction/convenience signal."),
    ("minimum_nights", "numeric_feature", "Listings", "Stay-rule signal."),
    ("minimum_nights_capped", "numeric_feature", "Engineered", "Robust stay-rule signal capped for extreme values."),
    ("maximum_nights", "numeric_feature", "Listings", "Stay-rule signal."),
    ("availability_30", "numeric_feature", "Listings", "Near-term availability signal."),
    ("availability_60", "numeric_feature", "Listings", "Short-term availability signal."),
    ("availability_90", "numeric_feature", "Listings", "Medium-term availability signal."),
    ("availability_365", "numeric_feature", "Listings", "Annual availability signal."),
    ("calendar_available_days", "numeric_feature", "Calendar", "Availability count from daily calendar."),
    ("calendar_unavailable_days", "numeric_feature", "Calendar", "Unavailable days; demand/host-usage proxy."),
    ("calendar_unavailable_rate", "numeric_feature", "Calendar", "Corrected listing-level availability/booking proxy."),
    ("calendar_unavailable_weekday", "numeric_feature", "Calendar", "Weekday availability/booking pattern."),
    ("calendar_unavailable_weekend", "numeric_feature", "Calendar", "Weekend availability/booking pattern."),
    ("calendar_unavailable_q1", "numeric_feature", "Calendar", "Quarterly seasonality proxy."),
    ("calendar_unavailable_q2", "numeric_feature", "Calendar", "Quarterly seasonality proxy."),
    ("calendar_unavailable_q3", "numeric_feature", "Calendar", "Quarterly seasonality proxy."),
    ("calendar_unavailable_q4", "numeric_feature", "Calendar", "Quarterly seasonality proxy."),
    ("calendar_unavailable_winter", "numeric_feature", "Calendar", "Seasonality proxy."),
    ("calendar_unavailable_spring", "numeric_feature", "Calendar", "Seasonality proxy."),
    ("calendar_unavailable_summer", "numeric_feature", "Calendar", "Seasonality proxy."),
    ("calendar_unavailable_autumn", "numeric_feature", "Calendar", "Seasonality proxy."),
    ("number_of_reviews", "numeric_feature", "Listings", "Review volume/reputation signal."),
    ("number_of_reviews_ltm", "numeric_feature", "Listings", "Recent annual review activity."),
    ("number_of_reviews_l30d", "numeric_feature", "Listings", "Very recent activity signal."),
    ("number_of_reviews_ly", "numeric_feature", "Listings", "Prior-year review activity where available."),
    ("reviews_per_month", "numeric_feature", "Listings", "Normalized review velocity."),
    ("has_reviews", "binary_feature", "Engineered from listings", "Distinguishes unrated/new listings."),
    ("days_since_last_review", "numeric_feature", "Engineered from listings", "Review recency signal."),
    ("review_history_days", "numeric_feature", "Engineered from listings", "Listing review-history maturity."),
    ("review_scores_rating", "numeric_feature", "Listings", "Overall quality/reputation signal."),
    ("review_scores_accuracy", "numeric_feature", "Listings", "Sub-rating quality signal."),
    ("review_scores_cleanliness", "numeric_feature", "Listings", "Sub-rating quality signal."),
    ("review_scores_checkin", "numeric_feature", "Listings", "Sub-rating service signal."),
    ("review_scores_communication", "numeric_feature", "Listings", "Sub-rating host-service signal."),
    ("review_scores_location", "numeric_feature", "Listings", "Guest perception of location."),
    ("review_scores_value", "numeric_feature", "Listings", "Guest perception of price/value."),
    ("raw_review_total_reviews", "numeric_feature_optional", "Reviews", "Raw review-table count, used only when keys match."),
    ("raw_reviews_last_6m", "numeric_feature_optional", "Reviews", "Raw review momentum, used only when keys match."),
]

This metadata dictionary explains each master-dataset attribute and is reused when the notebook writes the data dictionary output.

---
## 2. Audit Raw Review Compatibility

These helper functions prepare the output folder and select the corrected review file before checking whether review IDs match the listing IDs.

In [6]:
def ensure_output_dir() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
def ensure_review_file(city_key: str, config: dict[str, object]) -> Path:
    """Return the review source path, downloading known corrected files if missing."""
    path = RAW_DIR / str(config.get("reviews_file", f"{city_key}_reviews.csv"))
    if path.exists():
        return path

    url = config.get("reviews_url")
    if not url:
        raise FileNotFoundError(f"Review file not found: {path}")

    print(f"Review file missing for {config['city_label']}: {path.name}")
    print(f"Downloading compressed source from: {url}")
    compressed_path = path.with_suffix(path.suffix + ".gz")
    urllib.request.urlretrieve(str(url), compressed_path)
    with gzip.open(compressed_path, "rb") as src, path.open("wb") as dst:
        shutil.copyfileobj(src, dst)
    compressed_path.unlink(missing_ok=True)
    print(f"Saved: {path}")
    return path

In [8]:
for city_key, config in CITY_CONFIG.items():
    listing_path = RAW_DIR / f"{city_key}_listings.csv"
    reviews_path = ensure_review_file(city_key, config)

    listing_ids = set(pd.read_csv(listing_path, usecols=["id"])["id"].dropna().astype("int64"))
    review_ids = set(pd.read_csv(reviews_path, usecols=["listing_id"])["listing_id"].dropna().astype("int64"))
    matched = listing_ids & review_ids

    print(config["city_label"])
    print("  reviews file:", reviews_path.name)
    print("  matched listing IDs:", f"{len(matched):,}")
    print("  match rate:", f"{len(matched) / len(listing_ids) * 100:.2f}%")



Madrid
  reviews file: madrid_reviews_2025-09-14.csv
  matched listing IDs: 19,853
  match rate: 79.41%
Tokyo
  reviews file: tokyo_reviews.csv
  matched listing IDs: 24,009
  match rate: 85.92%


This checks whether each city's review file can actually join to the listings file. This step caught the original Madrid issue and confirms that the corrected Madrid reviews file now matches properly.


---
## 3. Clean Listing Attributes

These functions parse prices, booleans, rates, amenities, property groups, bathroom details, and distances before producing a clean listing-level table.

In [9]:
def parse_price_to_eur(price: pd.Series, currency: str) -> pd.Series:
    numeric = (
        price.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )
    numeric = pd.to_numeric(numeric, errors="coerce")
    if currency == "JPY":
        numeric = numeric * JPY_TO_EUR
    return numeric

In [10]:
def parse_boolean(series: pd.Series) -> pd.Series:
    return series.map({"t": 1, "f": 0, True: 1, False: 0}).astype("float")

In [11]:
def parse_rate(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace("%", "", regex=False).str.strip(),
        errors="coerce",
    ) / 100

In [12]:
def parse_amenities(value: object) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value)
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [str(item).strip().lower() for item in parsed]
    except (SyntaxError, ValueError):
        pass
    return [part.strip().strip('"').strip("'").lower() for part in text.split(",") if part.strip()]

In [13]:
def classify_property_type(value: object) -> str:
    text = str(value).lower()
    if "hotel" in text or "serviced" in text or "aparthotel" in text:
        return "Hotel/serviced"
    if "rental unit" in text or "apartment" in text or "condo" in text or "loft" in text:
        return "Apartment/condo"
    if "home" in text or "house" in text or "townhouse" in text or "villa" in text:
        return "House/villa"
    if "hostel" in text or "guesthouse" in text or "bed and breakfast" in text:
        return "Guesthouse/hostel"
    if "room" in text:
        return "Room"
    return "Other"

In [14]:
def bathroom_type(value: object) -> str:
    text = str(value).lower()
    if pd.isna(value) or text in {"nan", ""}:
        return "Unknown"
    if "shared" in text:
        return "Shared"
    if "private" in text:
        return "Private"
    if "half" in text:
        return "Half"
    return "Standard"

In [15]:
def parse_bathrooms(value: object) -> float:
    text = str(value).lower()
    if "half" in text and not any(char.isdigit() for char in text):
        return 0.5
    extracted = pd.Series([text]).str.extract(r"(\d+\.?\d*)").iloc[0, 0]
    return float(extracted) if pd.notna(extracted) else np.nan

In [16]:
def haversine_km(lat: pd.Series, lon: pd.Series, center_lat: float, center_lon: float) -> pd.Series:
    radius_km = 6371.0
    lat1 = np.radians(lat.astype(float))
    lon1 = np.radians(lon.astype(float))
    lat2 = math.radians(center_lat)
    lon2 = math.radians(center_lon)
    dlat = lat1 - lat2
    dlon = lon1 - lon2
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * math.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))

In [17]:
def keep_existing_columns(path: Path, columns: list[str]) -> list[str]:
    header = pd.read_csv(path, nrows=0).columns
    return [col for col in columns if col in header]

In [18]:
def load_and_clean_listings(city_key: str, config: dict[str, object]) -> tuple[pd.DataFrame, dict[str, object]]:
    path = RAW_DIR / f"{city_key}_listings.csv"
    usecols = keep_existing_columns(path, LISTING_COLUMNS)
    raw = pd.read_csv(path, usecols=usecols)
    audit = {"raw_listing_rows": len(raw)}

    df = raw.copy()
    df.rename(columns={"id": "listing_id"}, inplace=True)
    df["city"] = config["city_label"]
    df["price_eur"] = parse_price_to_eur(df["price"], str(config["currency"]))
    audit["missing_or_zero_price_rows"] = int(df["price_eur"].isna().sum() + (df["price_eur"] <= 0).sum())
    df = df[df["price_eur"].notna() & (df["price_eur"] > 0)].copy()

    q1 = df["price_eur"].quantile(0.25)
    q3 = df["price_eur"].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    before_outliers = len(df)
    df = df[(df["price_eur"] >= lower) & (df["price_eur"] <= upper)].copy()
    audit["price_iqr_lower_eur"] = round(float(lower), 4)
    audit["price_iqr_upper_eur"] = round(float(upper), 4)
    audit["price_outlier_rows_removed"] = int(before_outliers - len(df))

    df["log_price_eur"] = np.log1p(df["price_eur"])

    for col in BOOLEAN_COLUMNS:
        if col in df.columns:
            df[col] = parse_boolean(df[col])

    for col in ["host_response_rate", "host_acceptance_rate"]:
        if col in df.columns:
            df[col] = parse_rate(df[col])

    if "bathrooms_text" in df.columns:
        df["bathrooms"] = df["bathrooms_text"].apply(parse_bathrooms)
        df["bathroom_type"] = df["bathrooms_text"].apply(bathroom_type)

    amenities = df["amenities"].apply(parse_amenities) if "amenities" in df.columns else pd.Series([[]] * len(df))
    df["amenities_count"] = amenities.apply(len)
    for feature, terms in KEY_AMENITIES.items():
        df[feature] = amenities.apply(
            lambda values: int(any(term in amenity for amenity in values for term in terms))
        )

    df["property_group"] = df["property_type"].apply(classify_property_type)
    df["distance_to_center_km"] = haversine_km(
        df["latitude"],
        df["longitude"],
        float(config["center_lat"]),
        float(config["center_lon"]),
    ).round(4)
    df["is_central_5km"] = (df["distance_to_center_km"] <= 5).astype(int)

    neighbourhood_counts = df["neighbourhood_cleansed"].value_counts()
    df["neighbourhood_listing_count"] = df["neighbourhood_cleansed"].map(neighbourhood_counts)

    df["last_scraped"] = pd.to_datetime(df["last_scraped"], errors="coerce")
    df["host_since"] = pd.to_datetime(df["host_since"], errors="coerce")
    df["first_review"] = pd.to_datetime(df["first_review"], errors="coerce")
    df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

    df["host_tenure_days"] = (df["last_scraped"] - df["host_since"]).dt.days
    df["days_since_last_review"] = (df["last_scraped"] - df["last_review"]).dt.days
    df["review_history_days"] = (df["last_review"] - df["first_review"]).dt.days
    df["has_reviews"] = (df["number_of_reviews"].fillna(0) > 0).astype(int)

    df["minimum_nights_capped"] = df["minimum_nights"].clip(upper=30)
    df["is_professional_host"] = (df["host_listings_count"].fillna(0) >= 5).astype(int)

    df.drop(columns=["price", "amenities", "bathrooms_text"], inplace=True, errors="ignore")
    audit["clean_listing_rows_after_price_filter"] = len(df)
    return df, audit

---
## 4. Aggregate Calendar, Reviews, and Geodata

These functions turn the raw calendar and review files into listing-level features, then load neighbourhood geodata for location enrichment.

In [19]:
def season_from_month(month: pd.Series) -> pd.Series:
    return month.map(
        {
            12: "winter",
            1: "winter",
            2: "winter",
            3: "spring",
            4: "spring",
            5: "spring",
            6: "summer",
            7: "summer",
            8: "summer",
            9: "autumn",
            10: "autumn",
            11: "autumn",
        }
    )

In [20]:
def aggregate_calendar(city_key: str) -> pd.DataFrame:
    path = RAW_DIR / f"{city_key}_calendar.csv"
    partial_overall = []
    partial_quarter = []
    partial_season = []
    partial_weekend = []

    for chunk in pd.read_csv(
        path,
        usecols=["listing_id", "date", "available"],
        chunksize=CHUNK_SIZE,
    ):
        chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
        chunk["is_available"] = (chunk["available"] == "t").astype("int8")
        chunk["is_unavailable"] = 1 - chunk["is_available"]
        chunk["quarter"] = chunk["date"].dt.quarter.astype("Int64")
        chunk["season"] = season_from_month(chunk["date"].dt.month)
        chunk["is_weekend"] = chunk["date"].dt.dayofweek.ge(5).astype("int8")

        partial_overall.append(
            chunk.groupby("listing_id").agg(
                calendar_days=("is_available", "size"),
                calendar_available_days=("is_available", "sum"),
                calendar_unavailable_days=("is_unavailable", "sum"),
            )
        )
        partial_quarter.append(
            chunk.groupby(["listing_id", "quarter"]).agg(
                unavailable=("is_unavailable", "sum"),
                days=("is_unavailable", "size"),
            )
        )
        partial_season.append(
            chunk.groupby(["listing_id", "season"]).agg(
                unavailable=("is_unavailable", "sum"),
                days=("is_unavailable", "size"),
            )
        )
        partial_weekend.append(
            chunk.groupby(["listing_id", "is_weekend"]).agg(
                unavailable=("is_unavailable", "sum"),
                days=("is_unavailable", "size"),
            )
        )

    overall = pd.concat(partial_overall).groupby("listing_id").sum()
    overall["calendar_unavailable_rate"] = (
        overall["calendar_unavailable_days"] / overall["calendar_days"]
    )

    quarter = pd.concat(partial_quarter).groupby(["listing_id", "quarter"]).sum()
    quarter["rate"] = quarter["unavailable"] / quarter["days"]
    quarter_wide = (
        quarter["rate"]
        .unstack("quarter")
        .rename(columns=lambda q: f"calendar_unavailable_q{int(q)}")
    )

    season = pd.concat(partial_season).groupby(["listing_id", "season"]).sum()
    season["rate"] = season["unavailable"] / season["days"]
    season_wide = (
        season["rate"]
        .unstack("season")
        .rename(columns=lambda s: f"calendar_unavailable_{s}")
    )

    weekend = pd.concat(partial_weekend).groupby(["listing_id", "is_weekend"]).sum()
    weekend["rate"] = weekend["unavailable"] / weekend["days"]
    weekend_wide = weekend["rate"].unstack("is_weekend").rename(
        columns={0: "calendar_unavailable_weekday", 1: "calendar_unavailable_weekend"}
    )

    features = overall.join([quarter_wide, season_wide, weekend_wide]).reset_index()
    return features

In [21]:
def load_review_features(
    city_key: str,
    config: dict[str, object],
    listing_ids: set[int],
    reference_date: pd.Timestamp,
) -> tuple[pd.DataFrame, dict[str, object]]:
    path = ensure_review_file(city_key, config)
    reviews = pd.read_csv(path, usecols=["listing_id", "date"])
    reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")
    review_ids = set(reviews["listing_id"].dropna().astype("int64"))
    matched_ids = listing_ids & review_ids
    audit = {
        "reviews_source_file": path.name,
        "raw_review_rows": len(reviews),
        "raw_review_unique_listing_ids": len(review_ids),
        "raw_review_matched_listing_ids": len(matched_ids),
        "raw_review_listing_match_pct": round(len(matched_ids) / len(listing_ids) * 100, 4)
        if listing_ids
        else 0,
    }

    if len(matched_ids) == 0:
        return pd.DataFrame(columns=["listing_id", "raw_review_total_reviews", "raw_reviews_last_6m"]), audit

    recent_cutoff = reference_date - pd.DateOffset(months=6)
    agg = (
        reviews[reviews["listing_id"].isin(matched_ids)]
        .groupby("listing_id")
        .agg(
            raw_review_total_reviews=("date", "count"),
            raw_review_first_date=("date", "min"),
            raw_review_last_date=("date", "max"),
        )
        .reset_index()
    )
    recent = (
        reviews[(reviews["listing_id"].isin(matched_ids)) & (reviews["date"] >= recent_cutoff)]
        .groupby("listing_id")["date"]
        .count()
        .rename("raw_reviews_last_6m")
        .reset_index()
    )
    agg = agg.merge(recent, on="listing_id", how="left")
    agg["raw_reviews_last_6m"] = agg["raw_reviews_last_6m"].fillna(0)
    agg.drop(columns=["raw_review_first_date", "raw_review_last_date"], inplace=True)
    return agg, audit

In [22]:
def load_geodata(city_key: str) -> pd.DataFrame:
    path = GEODATA_DIR / f"{city_key}_neighbourhoods.csv"
    geo = pd.read_csv(path)
    if "neighbourhood" not in geo.columns:
        return pd.DataFrame(columns=["neighbourhood_cleansed", "neighbourhood_group"])
    geo = geo.rename(
        columns={
            "neighbourhood": "neighbourhood_cleansed",
            "neighbourhood_group": "neighbourhood_group",
        }
    )
    return geo[["neighbourhood_cleansed", "neighbourhood_group"]].drop_duplicates()

---
## 5. Assemble Master Datasets

These functions control the final column order, build each city master table, write the attribute dictionary, and save the recommended feature list.

In [23]:
def final_column_order(df: pd.DataFrame) -> list[str]:
    priority = [
        "listing_id",
        "city",
        "price_eur",
        "log_price_eur",
        "neighbourhood_cleansed",
        "neighbourhood_group",
        "latitude",
        "longitude",
        "distance_to_center_km",
        "is_central_5km",
        "neighbourhood_listing_count",
        "property_type",
        "property_group",
        "room_type",
        "accommodates",
        "bedrooms",
        "beds",
        "bathrooms",
        "bathroom_type",
        "amenities_count",
    ]
    amenity_cols = list(KEY_AMENITIES.keys())
    remaining = [col for col in df.columns if col not in priority + amenity_cols]
    return [col for col in priority if col in df.columns] + amenity_cols + remaining

In [24]:
def build_city_master(city_key: str, config: dict[str, object]) -> tuple[pd.DataFrame, dict[str, object]]:
    listings, audit = load_and_clean_listings(city_key, config)
    listing_ids = set(listings["listing_id"].dropna().astype("int64"))
    reference_date = listings["last_scraped"].max()

    calendar = aggregate_calendar(city_key)
    audit["calendar_unique_listing_ids"] = calendar["listing_id"].nunique()
    audit["calendar_matched_clean_listing_ids"] = int(calendar["listing_id"].isin(listing_ids).sum())
    audit["calendar_clean_listing_match_pct"] = round(
        audit["calendar_matched_clean_listing_ids"] / len(listing_ids) * 100,
        4,
    )

    reviews, review_audit = load_review_features(city_key, config, listing_ids, reference_date)
    audit.update(review_audit)

    geo = load_geodata(city_key)
    audit["geodata_neighbourhood_rows"] = len(geo)

    master = listings.merge(calendar, on="listing_id", how="left")
    master = master.merge(reviews, on="listing_id", how="left")
    master = master.merge(geo, on="neighbourhood_cleansed", how="left", suffixes=("", "_geo"))

    if "neighbourhood_group_cleansed" in master.columns:
        master["neighbourhood_group"] = master["neighbourhood_group_cleansed"].combine_first(
            master.get("neighbourhood_group")
        )
        master.drop(columns=["neighbourhood_group_cleansed"], inplace=True)

    audit["final_rows"] = len(master)
    audit["final_columns"] = master.shape[1]
    audit["price_median_eur"] = round(float(master["price_eur"].median()), 4)
    audit["price_mean_eur"] = round(float(master["price_eur"].mean()), 4)
    audit["calendar_unavailable_rate_unique_values"] = int(master["calendar_unavailable_rate"].nunique(dropna=True))
    audit["rows_missing_calendar_features"] = int(master["calendar_unavailable_rate"].isna().sum())
    audit["rows_missing_raw_review_features"] = int(master["raw_review_total_reviews"].isna().sum()) if "raw_review_total_reviews" in master.columns else len(master)

    master = master[final_column_order(master)]
    return master, audit

In [25]:
def write_attribute_dictionary() -> None:
    attr = pd.DataFrame(
        MODEL_ATTRIBUTE_META,
        columns=["attribute", "role", "source", "why_kept_or_note"],
    )
    attr.to_csv(OUTPUT_DIR / "master_model_attribute_dictionary.csv", index=False)

In [26]:
def write_recommended_feature_list(city_masters: dict[str, pd.DataFrame]) -> None:
    exclude = {
        "listing_id",
        "city",
        "price_eur",
        "log_price_eur",
        "last_scraped",
        "host_since",
        "first_review",
        "last_review",
        "calendar_days",
    }
    rows = []
    for city, df in city_masters.items():
        for col in df.columns:
            missing_pct = df[col].isna().mean() * 100
            unique_values = df[col].nunique(dropna=True)
            if col in exclude:
                recommendation = "exclude"
                note = "Identifier, target, raw date, constant city field, or non-informative calendar length."
            elif missing_pct > 35:
                recommendation = "exclude"
                note = "Too sparse for first-pass modelling; revisit only if city-specific handling is needed."
            elif unique_values <= 1:
                recommendation = "exclude"
                note = "No predictive variation within this city."
            else:
                recommendation = "use"
                note = "Usable city-specific predictor for first-pass models."

            rows.append(
                {
                    "city": city,
                    "attribute": col,
                    "recommendation": recommendation,
                    "missing_pct": round(missing_pct, 3),
                    "unique_values": int(unique_values),
                    "note": note,
                }
            )
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "recommended_model_features_by_city.csv", index=False)

In [27]:
def main() -> None:
    ensure_output_dir()
    audits = []
    city_masters = {}

    for city_key, config in CITY_CONFIG.items():
        print(f"\nBuilding master dataset for {config['city_label']}...")
        master, audit = build_city_master(city_key, config)
        city_masters[str(config["city_label"])] = master
        output_file = OUTPUT_DIR / f"{city_key}_master_model_dataset.csv"
        master.to_csv(output_file, index=False)
        audit["city"] = config["city_label"]
        audits.append(audit)
        print(f"  saved: {output_file}")
        print(f"  shape: {master.shape}")
        print(f"  median price EUR: {master['price_eur'].median():.2f}")

    audit_df = pd.DataFrame(audits)
    audit_df = audit_df[["city"] + [col for col in audit_df.columns if col != "city"]]
    audit_df.to_csv(OUTPUT_DIR / "master_dataset_audit.csv", index=False)
    write_attribute_dictionary()
    write_recommended_feature_list(city_masters)

    print(f"\nAudit and attribute dictionary saved to: {OUTPUT_DIR}")
    print("\nAudit summary")
    print(audit_df.to_string(index=False))

---
## 6. Build Madrid and Tokyo Master Datasets

In [28]:
main()



Building master dataset for Madrid...
  saved: C:\Users\georg\Desktop\Docs\Student Files\CAPSTONE\1. Data\master\madrid_master_model_dataset.csv
  shape: (17770, 88)
  median price EUR: 105.00

Building master dataset for Tokyo...
  saved: C:\Users\georg\Desktop\Docs\Student Files\CAPSTONE\1. Data\master\tokyo_master_model_dataset.csv
  shape: (23765, 88)
  median price EUR: 95.91

Audit and attribute dictionary saved to: C:\Users\georg\Desktop\Docs\Student Files\CAPSTONE\1. Data\master

Audit summary
  city  raw_listing_rows  missing_or_zero_price_rows  price_iqr_lower_eur  price_iqr_upper_eur  price_outlier_rows_removed  clean_listing_rows_after_price_filter  calendar_unique_listing_ids  calendar_matched_clean_listing_ids  calendar_clean_listing_match_pct           reviews_source_file  raw_review_rows  raw_review_unique_listing_ids  raw_review_matched_listing_ids  raw_review_listing_match_pct  geodata_neighbourhood_rows  final_rows  final_columns  price_median_eur  price_mean_eur  c

This runs the full master-dataset build. It cleans listings, aggregates calendar availability, joins corrected review features, joins geodata, and writes one master CSV per city.


---
## 4. Review Audit Output


In [29]:
audit_path = OUTPUT_DIR / "master_dataset_audit.csv"
audit = pd.read_csv(audit_path)
audit


,city,raw_listing_rows,missing_or_zero_price_rows,price_iqr_lower_eur,price_iqr_upper_eur,price_outlier_rows_removed,clean_listing_rows_after_price_filter,calendar_unique_listing_ids,calendar_matched_clean_listing_ids,calendar_clean_listing_match_pct,...,raw_review_matched_listing_ids,raw_review_listing_match_pct,geodata_neighbourhood_rows,final_rows,final_columns,price_median_eur,price_mean_eur,calendar_unavailable_rate_unique_values,rows_missing_calendar_features,rows_missing_raw_review_features
0,Madrid,25000,6047,-71.0000,305.0000,1183,17770,25000,17770,100.0,...,14945,84.1024,128,17770,88,105.000,114.5985,373,0,2825
1,Tokyo,27945,2465,-59.2953,284.0236,1715,23765,27945,23765,100.0,...,20662,86.9430,62,23765,88,95.914,109.8130,377,0,3103


This audit table documents row counts, price filtering, calendar matches, review matches, and final dataset sizes. It is useful for reporting and defending the data preparation process.


---
## 5. Inspect Attributes and Files


In [30]:
attribute_dictionary = pd.read_csv(OUTPUT_DIR / "master_model_attribute_dictionary.csv")
attribute_dictionary.head(20)


,attribute,role,source,why_kept_or_note
0,listing_id,identifier,Listings,Audit/join key only; do not use as a model pre...
1,city,context,Listings,Identifies the city; constant within each city...
2,price_eur,target,Listings,Nightly listing price converted to EUR; target...
3,log_price_eur,target_transform,Engineered,Log target option for skewed price distributions.
4,neighbourhood_cleansed,categorical_feature,Listings/geodata,Captures local market/location effects.
5,neighbourhood_group,categorical_feature,Listings/geodata,Broader location grouping where available.
6,latitude,numeric_feature,Listings,Fine-grained location signal.
7,longitude,numeric_feature,Listings,Fine-grained location signal.
8,distance_to_center_km,numeric_feature,Engineered,Proxy for centrality and tourist/business access.
9,is_central_5km,binary_feature,Engineered,Simple central-vs-noncentral location flag.


This dictionary explains what each master-dataset attribute is for and why it was kept. It can be reused later for the report and the chatbot design.


In [31]:
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(path.name)


airbnb_clean_final.csv
airbnb_madrid_clean.csv
airbnb_tokyo_clean.csv
madrid_master_model_dataset.csv
master_dataset_audit.csv
master_model_attribute_dictionary.csv
recommended_model_features_by_city.csv
tokyo_master_model_dataset.csv


This lists the output files created by the master-dataset workflow.
